# Aspect Term Extraction with BERT (laptop reviews)

## What is this notebook for?

When people review a product they talk about specific **parts or features** of it:

> "The **battery life** is great but the **keyboard** feels cheap."

The things being judged here, *battery life* and *keyboard*, are called **aspect terms**. Finding them automatically is called **Aspect Term Extraction (ATE)**. It is the first half of *Aspect-Based Sentiment Analysis*: first you find **what** the reviewer is talking about (this notebook), and later you can decide whether they were **positive or negative** about each thing (not done here).

This notebook teaches a computer to do that. It takes **BERT**, a language model that has already "read" a huge amount of English text, and **fine-tunes** it (trains it a little more) on laptop reviews in which humans have already marked the aspect terms. The reviews come from the SemEval-2014 Task 4 laptop dataset.

## How is the problem expressed? (BIO tags)

The model does not output phrases directly. Instead it gives **one label to every word**:

| Label | Meaning | Number used inside the model |
|---|---|---|
| `B` | **B**eginning: the first word of an aspect term | 1 |
| `I` | **I**nside: a later word of the same aspect term | 2 |
| `O` | **O**utside: the word is not part of any aspect term | 0 |

Example:

| Word | The | battery | life | is | great |
|---|---|---|---|---|---|
| Label | O | B | I | O | O |

Reading off the `B` + following `I` words gives the aspect term "battery life". Giving a label to every word (token) is called **token classification**; it is the same technique used for Named Entity Recognition (NER), which is why the code uses names such as `ner_tags` and `task = "ner"`.

## What the notebook does, step by step

1. **Install and import** the libraries.
2. **Load the data** (`train.json`, `test.json`) and look at it.
3. **Tokenize**: convert the words into the numbers BERT understands, and line the labels up with them.
4. **Train** (fine-tune) BERT and measure how good it is after each pass over the data.
5. **Save** the trained model to disk.
6. **Predict**: load the saved model and label every sentence of the test set, saving the predictions to a file.
7. **Score**: compute an F1 score and a confusion matrix for those predictions.

## What you need before running

- `train.json` and `test.json` in the **same folder** as this notebook.
- An internet connection the first time (to download BERT, about 440 MB).
- A GPU is strongly recommended. Training takes a couple of minutes on a GPU and much longer on a CPU.

## What the notebook creates

| File / folder | What it is |
|---|---|
| `test-ner/` | Training checkpoints (intermediate saves) written automatically during training |
| `ae_laptop_bert.model/` | The final trained model and its tokenizer (a folder, despite the name) |
| `laptop_bert_base_preds.json` | The model's predicted labels for every test sentence |

## Mini glossary

- **Model**: a mathematical function with millions of adjustable numbers ("weights"). Training means adjusting them.
- **BERT**: a pre-trained language model from Google. `bert-base-uncased` is the standard-size version that ignores capital letters.
- **Fine-tuning**: continuing the training of a pre-trained model on your own, smaller, task-specific data.
- **Token**: the small text unit a model works with. For BERT a token is a word or a *piece* of a word.
- **Epoch**: one complete pass over the whole training set.
- **Batch**: a small group of examples processed together (here 16 sentences).
- **Logits**: the raw scores a model outputs, one per possible label. The highest score is the model's choice.

Run the cells **in order, from top to bottom**. Each cell relies on the ones before it.

In [ ]:
# ---------------------------------------------------------------------------
# STEP 1a: Install the libraries this notebook needs.
#
# A line that starts with "!" is NOT Python. It is a command sent to the
# operating system's terminal, exactly as if you typed it in a console.
# "pip install X" downloads the Python package X from the internet and
# installs it. If a package is already installed, pip just says
# "Requirement already satisfied" and moves on, so re-running is harmless.
# ---------------------------------------------------------------------------

# transformers: the Hugging Face library that contains BERT, its tokenizer,
# and the "Trainer" tool that runs the training loop for us.
!pip install transformers

# datasets: the Hugging Face library for holding training data in an
# efficient table-like structure that the Trainer understands.
!pip install datasets

# seqeval: a small library that scores sequence-labelling tasks (B/I/O tags).
# It knows how to compare whole aspect terms, not just single words.
!pip install seqeval

# evaluate: the Hugging Face library that gives a common interface to many
# metrics. We use it to load the seqeval metric above.
# (seqeval is listed a second time here; that is harmless, just redundant.)
!pip install evaluate seqeval

In [ ]:
# ---------------------------------------------------------------------------
# STEP 1b: First imports.
#
# "import" makes an installed library available in this notebook.
# ---------------------------------------------------------------------------

# pandas is the standard Python library for tables of data (called DataFrames).
# "as pd" gives it the short nickname "pd", so we can write pd.read_json(...)
# instead of pandas.read_json(...).
import pandas as pd

# The Hugging Face transformers library (BERT lives in here).
# This notebook has since been updated to work with current versions.
import transformers

In [ ]:
# Show which version of transformers is actually installed.
# In a notebook, when the LAST line of a cell is just a value (no "print"),
# Jupyter displays that value underneath the cell.
# This is useful to know because function and argument names changed between
# versions (for example "evaluation_strategy" became "eval_strategy").
transformers.__version__

## Step 2: Load the data and look at it

The data files are in JSON format. Inside, every sentence has a number and two lists of equal length:

```
{"0": {"label":    ["B",    "I",    "O",  "O",     "O"],
       "sentence": ["Boot", "time", "is", "super", "fast"]},
 "1": {...},
 ...}
```

`sentence` holds the words, `label` holds the B/I/O tag of each word.

In [ ]:
# Take a first look at the test data, just for us humans. Nothing from this
# cell is stored in a variable or used later.
#
# pd.read_json('test.json')
#     Reads the file into a table. Because of how the file is organised,
#     pandas puts the SENTENCES in the columns (0, 1, 2, ...) and the two
#     fields ("label", "sentence") in the rows. That is sideways for us.
#
# .transpose()
#     Flips the table: rows become columns and columns become rows.
#     Now every ROW is one sentence, and the two COLUMNS are
#     "label" and "sentence". This is the usual, readable layout.
#
# If you get "FileNotFoundError", test.json is not in the same folder
# as this notebook.
pd.read_json('test.json').transpose()

In [ ]:
# ---------------------------------------------------------------------------
# Imports: bring in everything the rest of the notebook needs.
# ---------------------------------------------------------------------------

# numpy: the standard library for fast maths on arrays of numbers.
import numpy as np

# Dataset: the Hugging Face table type that the Trainer expects as input.
from datasets import Dataset

# evaluate: used later to load the "seqeval" scoring metric.
import evaluate

# AutoTokenizer: loads the right tokenizer for a given model name.
# A tokenizer turns text into the numbers a model can read.
from transformers import AutoTokenizer

# AutoModelForTokenClassification: loads BERT with an extra final layer that
#     outputs one label per token (exactly what B/I/O tagging needs).
# TrainingArguments: a container for all training settings.
# Trainer: the tool that actually runs the training loop for us.
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

# DataCollatorForTokenClassification: glues several sentences of different
# lengths into one batch by padding the short ones (explained further down).
from transformers import DataCollatorForTokenClassification

# torch (PyTorch): the deep-learning library that transformers is built on.
import torch

# ---------------------------------------------------------------------------
# The labels.
# ---------------------------------------------------------------------------

# The three possible labels, in a list. The POSITION in the list is the number
# the model uses: 'O' is 0, 'B' is 1, 'I' is 2. So label_list[1] gives 'B'.
# We use this later to turn the model's numbers back into letters.
label_list = ['O','B','I']

# The same mapping in the opposite direction: letter -> number.
# A dictionary lets us look up label_encoding_dict['B'] and get 1.
# Models can only work with numbers, never with letters.
# IMPORTANT: this must agree with label_list above (O=0, B=1, I=2). It does.
label_encoding_dict = {'I': 2, 'O': 0, 'B': 1,}

# ---------------------------------------------------------------------------
# Settings.
# ---------------------------------------------------------------------------

# Just a name. It is only used to name the output folder ("test-ner").
# "ner" = Named Entity Recognition, the classic task that uses this same
# one-label-per-word technique.
task = "ner" 

# Which pre-trained model to download from the Hugging Face Hub.
# "bert-base-uncased" = the standard-size BERT (about 110 million weights),
# "uncased" meaning it lowercases everything ("Laptop" and "laptop" are equal).
model_checkpoint = "bert-base-uncased"

# How many sentences the model looks at in one go. Bigger = faster but needs
# more GPU memory. If you get an "out of memory" error, make this smaller (8).
batch_size = 16
    
# Download (first time only) and load the tokenizer that belongs to this BERT.
# The tokenizer and the model MUST match, because the model was trained on the
# numbers produced by exactly this tokenizer.
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# ---------------------------------------------------------------------------
# Helper function 1: read one JSON file into a tidy table.
# The parameter is called "directory" but it is really the path of a FILE.
# ---------------------------------------------------------------------------
def get_all_tokens_and_ner_tags(directory):
    # One long chained line; read it left to right:
    #   pd.read_json(directory)  -> read the file (table comes out sideways)
    #   .transpose()             -> flip it: one row per sentence
    #   .reset_index()           -> the row names ("0","1",...) become an
    #                               ordinary column called 'index', and the
    #                               rows get fresh numbers 0, 1, 2, ...
    #   .drop('index', axis=1)   -> throw that 'index' column away, we do not
    #                               need it. axis=1 means "a column"
    #                               (axis=0 would mean "a row").
    df = pd.read_json(directory).transpose().reset_index().drop('index', axis=1)

    # Rename the two columns to the names used in the rest of the code:
    #   "sentence" -> "tokens"   (the list of words)
    #   "label"    -> "ner_tags" (the list of B/I/O letters)
    df = df.rename(columns={"sentence": "tokens", "label": "ner_tags"})

    # Hand the finished table back to whoever called the function.
    return df

  
# ---------------------------------------------------------------------------
# Helper function 2: load BOTH files and convert them to Hugging Face Datasets.
# ("un_token" = un-tokenized: the data is still plain words, not yet numbers.)
# ---------------------------------------------------------------------------
def get_un_token_dataset(train_directory, test_directory):
    # Read each file into a pandas table using helper function 1.
    train_df = get_all_tokens_and_ner_tags(train_directory)
    test_df = get_all_tokens_and_ner_tags(test_directory)

    # Convert the pandas tables into Hugging Face Dataset objects. Same
    # content, different container: the Trainer wants this type.
    train_dataset = Dataset.from_pandas(train_df)
    test_dataset = Dataset.from_pandas(test_df)

    # Return both at once (as a pair).
    return (train_dataset, test_dataset)

# Actually CALL the function now, and store the two results.
#   train_dataset: the sentences the model LEARNS from.
#   test_dataset:  sentences the model never sees during learning. We use them
#                  to check honestly how well it does on new text.
train_dataset, test_dataset = get_un_token_dataset('train.json', 
                                                   'test.json')

## Step 3: Tokenize, and line the labels up with the tokens

**The problem.** BERT does not read words. It reads **tokens**, and it has a fixed vocabulary of about 30,000 of them. A word that is not in the vocabulary is cut into known pieces (pieces after the first start with `##`). BERT also adds two special tokens: `[CLS]` at the start and `[SEP]` at the end of every sentence.

Our labels, however, are **one per word**. After tokenizing there are more tokens than labels, so we must build a new label list with **one label per token**:

| | | | | | | |
|---|---|---|---|---|---|---|
| Words | | touchscreen | | works | | |
| Word label | | B | | O | | |
| Tokens | `[CLS]` | touch | ##screen | works | `[SEP]` | |
| Belongs to word no. | none | 0 | 0 | 1 | none | |
| Token label | -100 | 1 (B) | 1 (B) | 0 (O) | -100 | |

**The special value -100** means "ignore this position". PyTorch's loss function skips every position labelled -100, so the model is neither rewarded nor punished for whatever it says about `[CLS]` and `[SEP]`.

(The exact way a word is split depends on BERT's vocabulary; the split shown is an illustration.)

In [ ]:
# This function receives a BATCH of examples (many sentences at once) and
# returns them tokenized, with a correctly aligned "labels" list for each.
def tokenize_and_align_labels(examples):
    # A switch that decides what to do with words split into several pieces.
    #   True  -> EVERY piece gets the word's label (used here).
    #   False -> only the FIRST piece gets the label, the others get -100
    #            (ignored).
    # Side effect of True: a 'B' word split in two pieces gives "B B"
    # instead of "B I". It is a simplification the original author chose.
    label_all_tokens = True

    # Run the tokenizer on all sentences of the batch.
    #   list(examples["tokens"])   the sentences; each is a list of words.
    #   is_split_into_words=True   tells the tokenizer "the text is ALREADY
    #                              cut into words, do not split on spaces
    #                              yourself". This is what lets us find out
    #                              later which word each token came from.
    #   truncation=True            if a sentence is longer than BERT's limit
    #                              (512 tokens) cut off the extra tokens
    #                              instead of crashing.
    # The result contains, for every sentence:
    #   input_ids       the token numbers
    #   attention_mask  1 for every real token (used later with padding)
    #   token_type_ids  all zeros here (only matters for sentence PAIRS)
    tokenized_inputs = tokenizer(list(examples["tokens"]), truncation=True, is_split_into_words=True)

    # Here we will collect the new, token-level label list of every sentence.
    labels = []

    # Go through the sentences one at a time.
    # enumerate gives us both a counter (i = 0, 1, 2, ...) and the item
    # (label = that sentence's list of letters such as ['B','I','O']).
    for i, label in enumerate(examples["ner_tags"]):
        # For sentence number i: for EACH TOKEN, which WORD did it come from?
        # Example: [None, 0, 0, 1, None]
        #   None = special token ([CLS] or [SEP]), belongs to no word
        #   0, 0 = two pieces of word number 0
        #   1    = word number 1
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        # Remembers the word number of the previous token, so we can tell
        # "first piece of a word" from "later piece of the same word".
        previous_word_idx = None

        # The token-level labels (as numbers) for THIS sentence.
        label_ids = []

        # Go through the tokens of this sentence one by one.
        for word_idx in word_ids:
            # Case 1: special token ([CLS] / [SEP]).
            # Give it -100 so the training ignores it.
            if word_idx is None:
                label_ids.append(-100)

            # Case 2: the word's label is the character '0' (the DIGIT zero).
            # Our data uses the LETTER 'O' (capital o), never the digit, so
            # this branch may never  run. It is a safety net
            elif label[word_idx] == '0':
                label_ids.append(0)

            # Case 3: this token is the FIRST piece of a new word.
            # Look up the word's letter and convert it to its number
            # ('O'->0, 'B'->1, 'I'->2).
            elif word_idx != previous_word_idx:
                label_ids.append(label_encoding_dict[label[word_idx]])

            # Case 4: this token is a LATER piece of the same word.
            # Because label_all_tokens is True it gets the word's label too.
            # (If it were False it would get -100 and be ignored.)
            else:
                label_ids.append(label_encoding_dict[label[word_idx]] if label_all_tokens else -100)

            # Remember this token's word number for the next round.
            previous_word_idx = word_idx

        # This sentence is finished: store its labels.
        labels.append(label_ids)
        
    # Add the aligned labels to the tokenizer's output under the name
    # "labels". The Trainer looks for exactly this name to know the
    # correct answers.
    tokenized_inputs["labels"] = labels
    return tokenized_inputs


# Apply the function to every sentence of both datasets.
#   .map(function)  runs the function over the dataset and adds the columns
#                   it returns (input_ids, attention_mask, labels, ...).
#   batched=True    hands the function many sentences at once (1000 by
#                   default) instead of one by one, which is much faster.
# The original columns ("tokens", "ner_tags") are kept alongside the new ones.
train_tokenized_datasets = train_dataset.map(tokenize_and_align_labels, batched=True)
test_tokenized_datasets = test_dataset.map(tokenize_and_align_labels, batched=True)

## Step 4: Train (fine-tune) the model

In this step we:

1. load pre-trained BERT with a new, untrained output layer for our 3 labels,
2. choose the training settings,
3. define how the model is scored,
4. run the training, score the result, and save the model.

**How training works, in one paragraph.** The model is shown a batch of sentences and predicts a label for every token. Its predictions are compared with the correct labels, giving a single number called the **loss** (how wrong it was). Then every weight in the model is nudged slightly in the direction that would have made the loss smaller. This is repeated for batch after batch. The `Trainer` does all of it for us.

**How the model is scored.** We use three standard numbers, all between 0 and 1 (higher is better):

- **Precision**: of the aspect terms the model *claimed* to find, what fraction were correct?
- **Recall**: of the aspect terms that *really exist*, what fraction did the model find?
- **F1**: a single number combining the two (their harmonic mean). It is only high when *both* are high.

**While it runs** you will see a "LOAD REPORT" listing some weights as UNEXPECTED and MISSING. That is normal: the parts of BERT used only for its original pre-training are thrown away (UNEXPECTED), and our brand-new output layer `classifier` starts with random numbers (MISSING) because it has not been trained yet. Training it is the whole point of this step.

In [ ]:
# ---------------------------------------------------------------------------
# 4.1 Load the model.
# ---------------------------------------------------------------------------
# Download (first time only) pre-trained BERT and put a fresh "token
# classification head" on top: one small extra layer that turns BERT's
# understanding of each token into scores for our labels.
#   num_labels=len(label_list)  -> 3, because we have 3 labels (O, B, I).
# So for every token the model will output 3 numbers (one score per label).
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=len(label_list))

# ---------------------------------------------------------------------------
# 4.2 Training settings.
# ---------------------------------------------------------------------------
args = TrainingArguments(
    # Folder where checkpoints (intermediate saves) are written.
    # f"test-{task}" inserts the variable task into the text -> "test-ner".
    f"test-{task}",

    # Measure the quality on the test set at the end of EVERY epoch.
    # (This argument was called "evaluation_strategy" in older versions.)
    eval_strategy = "epoch",

    # Learning rate: how big each correction step is. 1e-4 means 0.0001.
    # Too big -> learning is unstable; too small -> learning is very slow.
    # 1e-4 is on the high side for BERT (2e-5 to 5e-5 is more usual).
    learning_rate=1e-4,

    # Number of sentences per batch while training (16, set earlier).
    per_device_train_batch_size=batch_size,

    # Number of sentences per batch while evaluating.
    per_device_eval_batch_size=batch_size,

    # Go through the whole training set 2 times.
    num_train_epochs=2,

    # Weight decay: a gentle pull of the weights towards zero, which helps
    # prevent the model from simply memorising the training sentences.
    # 1e-5 (0.00001) is a very small amount.
    weight_decay=1e-5,
)

# ---------------------------------------------------------------------------
# 4.3 The data collator (the "batch builder").
# ---------------------------------------------------------------------------
# Sentences have different lengths, but a batch must be a neat rectangle of
# numbers. The collator takes the sentences of a batch and PADS the shorter
# ones with filler tokens up to the length of the longest one. It also pads
# the labels with -100, so the filler positions are ignored in training.
data_collator = DataCollatorForTokenClassification(tokenizer)

# Load the seqeval metric. It scores at the level of WHOLE aspect terms:
# a predicted term only counts as correct if it starts and ends exactly
# where the true one does ("battery" alone is wrong for "battery life").
metric = evaluate.load("seqeval")


# ---------------------------------------------------------------------------
# 4.4 The scoring function. The Trainer calls it automatically after each
#     evaluation and hands it the predictions and the correct answers.
# ---------------------------------------------------------------------------
def compute_metrics(p):
    # p is a pair: (the model's output, the correct labels). Unpack it.
    predictions, labels = p

    # predictions holds, for every sentence and every token, 3 scores
    # ("logits"), one per label. Its shape is
    #     (number of sentences, number of tokens, 3).
    # np.argmax(..., axis=2) picks, along the last direction (the 3 scores),
    # the POSITION of the biggest score: 0, 1 or 2. That is the predicted
    # label number. Afterwards the shape is (sentences, tokens).
    predictions = np.argmax(predictions, axis=2)

    # Convert the predicted NUMBERS back into LETTERS, and skip every
    # position whose correct label is -100 (special tokens and padding).
    # So... read from the right:
    #   for prediction, label in zip(predictions, labels)
    #       -> take one sentence at a time (its predictions + its labels)
    #   for (p, l) in zip(prediction, label) if l != -100
    #       -> take one token at a time, keep it only if it is not ignored
    #   label_list[p]
    #       -> turn the number p into its letter (0->'O', 1->'B', 2->'I')
    # zip(a, b) simply walks through two lists side by side, in pairs.
    true_predictions = [[label_list[p] for (p, l) in zip(prediction, label) if l != -100] for prediction, label in zip(predictions, labels)]

    # The same thing for the CORRECT labels (note label_list[l], not [p]).
    true_labels = [[label_list[l] for (p, l) in zip(prediction, label) if l != -100] for prediction, label in zip(predictions, labels)]

    # Let seqeval compare the predicted letters with the correct letters.
    results = metric.compute(predictions=true_predictions, references=true_labels)

    # Return the four numbers we care about, in a dictionary. The Trainer
    # shows them in the progress table (prefixed with "eval_").
    #   precision: of the terms the model found, how many were right
    #   recall:    of the real terms, how many the model found
    #   f1:        combination of precision and recall
    #   accuracy:  fraction of tokens with the right label. Be careful: most
    #              tokens are 'O', so accuracy looks high even for a weak
    #              model. F1 is the number to trust.
    return {"precision": results["overall_precision"], "recall": results["overall_recall"], 
            "f1": results["overall_f1"], "accuracy": results["overall_accuracy"]}
    
# ---------------------------------------------------------------------------
# 4.5 Build the Trainer: put all the pieces together.
# ---------------------------------------------------------------------------
trainer = Trainer(
    # The model to train.
    model,
    # The settings defined above.
    args,
    # The data to learn from.
    train_dataset=train_tokenized_datasets,
    # The data used to measure quality after each epoch.
    eval_dataset=test_tokenized_datasets,
    # The batch builder (padding).
    data_collator=data_collator,
    # The tokenizer, so that it is saved together with the model.
    # (This argument was called "tokenizer" in older versions.)
    processing_class=tokenizer,
    # Our scoring function.
    compute_metrics=compute_metrics
)

# ---------------------------------------------------------------------------
# 4.6 Run it.
# ---------------------------------------------------------------------------
# Start the training. The Trainer automatically uses a GPU if one is
# available. A table appears showing the loss and our four scores per epoch.
# "Training Loss" should go DOWN and F1 should go UP.
trainer.train()

# Evaluate once more on the test set. Because this is not the last line of
# the cell, its result is not displayed underneath; the same numbers are
# already visible in the last row of the training table.
trainer.evaluate()

# Save the trained model (and the tokenizer) into a FOLDER called
# "ae_laptop_bert.model" next to this notebook. Despite the ".model" ending
# it is a folder, not a single file. ("ae" = aspect extraction.)
trainer.save_model('ae_laptop_bert.model')

## Step 5: Predict with the saved model

Training is finished. Now we do what you would do in real use: **load the saved model from disk** and ask it to label sentences. Here we label every sentence of the test set and save the answers to a file.

Loading from disk (instead of just reusing the `model` variable from above) proves that the saved folder really contains everything needed, and it means you can run this part later without training again. To do that, first run the import cell and the tokenization cell above, because this part uses `label_list`, `torch`, `np` and `test_tokenized_datasets`.

In [ ]:
# Load the tokenizer from the folder we saved in the previous step.
# "./" means "the current folder" (where this notebook is).
# It is the same tokenizer as before; it was saved together with the model.
tokenizer = AutoTokenizer.from_pretrained('./ae_laptop_bert.model/')

In [ ]:
# Load OUR fine-tuned model from the saved folder (not the original BERT
# from the internet). This replaces the "model" variable from training.
# A model loaded this way sits on the CPU, which is why the next cells
# use .to('cpu') for the data: model and data must be on the same device.
# (num_labels=3 is already stored in the saved folder; passing it again
# is harmless.)
model = AutoModelForTokenClassification.from_pretrained('./ae_laptop_bert.model/', num_labels=len(label_list))

In [ ]:
# Two empty lists to collect things in:
#   results: the model's predicted labels, one inner list per sentence
#   labels:  the correct labels, one inner list per sentence
results = []
labels = []

# Switch the model to "evaluation mode". During training, parts of the model
# are randomly switched off on purpose (dropout). eval()
# turns that off, so the model gives its best, repeatable answers.
model.eval()

# Inside this block PyTorch does not record the information it would need
# for training. We are only predicting, so this saves memory and time.
with torch.no_grad():
    # Go through the test set ONE sentence at a time. "i" is one example:
    # a dictionary with keys such as 'input_ids', 'attention_mask', 'labels'.
    for i in test_tokenized_datasets:
        # The next statement does a lot in one go. From the inside out:
        #
        # torch.tensor(i['input_ids'])
        #     Turn the list of token numbers into a PyTorch tensor (PyTorch's
        #     array type; the model only accepts tensors).
        # .unsqueeze(0)
        #     Add a "batch" dimension at the front. The model always expects
        #     a batch (a stack of sentences), so one sentence of N tokens
        #     must be reshaped from (N) to (1, N): a batch containing 1.
        # .to('cpu')
        #     Make sure the data is on the CPU, where the model is.
        # torch.tensor(i['attention_mask'])...
        #     The same for the attention mask (1 = real token). It is all
        #     ones here because a single sentence needs no padding.
        # model.forward(input_ids, attention_mask)
        #     Run the model. (Writing model(...) does the same thing.)
        # [0]
        #     The model returns several things; the first is the "logits":
        #     the raw scores, with shape (1, N, 3).
        # [0]
        #     Take the first (and only) sentence of the batch -> (N, 3).
        # .numpy()
        #     Convert from a PyTorch tensor to a numpy array.
        # [1:-1]
        #     Drop the first and the last row, i.e. the [CLS] and [SEP]
        #     special tokens. ("1:-1" = from position 1 up to, not
        #     including, the last.) Shape is now (N-2, 3).
        # np.argmax(..., 1)
        #     For each remaining token pick the position of the highest of
        #     its 3 scores: 0 (O), 1 (B) or 2 (I). That is the prediction.
        # list(...)
        #     Turn the result into a plain Python list.
        res = list(np.argmax(model.forward(torch.tensor(i['input_ids']).unsqueeze(0).to('cpu'), 
                      torch.tensor(i['attention_mask']).unsqueeze(0).to('cpu'))[0][0].numpy()[1:-1],1))

        # The numbers in res are numpy integers. Convert each to an ordinary
        # Python int, because the JSON file format (used below to save the
        # predictions) cannot store numpy integers.
        resint = [int(arg) for arg in res]

        # Store this sentence's predictions as one entry of results.
        results.append(resint)

        # Store the correct labels too, again without the first and last
        # positions (the -100 of [CLS] and [SEP]).
        # (This "labels" list is not used when saving; the scoring section
        # below builds its own.)
        labels.append(list(np.array(i['labels'][1:-1])))

In [ ]:
# json: Python's built-in library for reading and writing JSON, a simple
# text format for storing lists and dictionaries in a file.
import json

In [ ]:
# Save the predictions to a file so they can be used later (for example by
# a second model that decides the sentiment of each aspect term).
#
# json.dumps(...) turns a Python object into JSON text. The object here is a
# dictionary with ONE key, "laptop_bert_base_preds", whose value is the list
# of predictions: one inner list per test sentence, one number per token
# (0 = O, 1 = B, 2 = I).
# Remember: the numbers are per TOKEN (word piece), not per original word,
# so a sentence can have more predictions than it has words.
json_string = json.dumps({"laptop_bert_base_preds":results})

# Open (create) the file for writing ('w'; an existing file is overwritten).
# The "with" block closes the file automatically when it ends.
with open('laptop_bert_base_preds.json', 'w') as outfile:
    # Write the JSON text into the file.
    outfile.write(json_string)

## Step 6: Score the predictions (F1 score and confusion matrix)

Here the predictions are scored a second time, in a simpler and stricter-to-read way than during training:

- During training, `seqeval` scored **whole aspect terms** (a term is right only if it matches exactly from first to last word).
- Here, `scikit-learn` scores **token by token**: every single token is either labelled correctly or not.

Because these are two different ways of counting, **the F1 here will not be equal to the F1 in the training table**. Both are valid; they just answer slightly different questions.

In [ ]:
# Predict the whole test set again, exactly as in the prediction cell above,
# with ONE difference: "+=" instead of ".append".
#   .append(x) adds x as ONE new entry          -> a list of lists
#   += x       adds every item of x separately  -> one long FLAT list
# The scoring functions below want two flat lists of equal length:
#   results: [predicted label of token 1, of token 2, ...] over ALL sentences
#   labels:  [correct label of token 1, of token 2, ...]   over ALL sentences
# We no longer care which sentence a token belonged to.
results = []
labels = []

# Evaluation mode (dropout off), as before.
model.eval()

# No training bookkeeping needed, as before.
with torch.no_grad():
    # One test sentence at a time.
    for i in test_tokenized_datasets:
        # Same chain as before: make tensors, add the batch dimension, run
        # the model, take the scores, drop [CLS] and [SEP], pick the best
        # label for every token. Then add them all to the flat list.
        results+=list(np.argmax(model.forward(torch.tensor(i['input_ids']).unsqueeze(0).to('cpu'), 
                      torch.tensor(i['attention_mask']).unsqueeze(0).to('cpu'))[0][0].numpy()[1:-1],1))

        # The correct labels of the same tokens (again without the first and
        # last position), added to the other flat list.
        labels+=list(np.array(i['labels'][1:-1]))

In [ ]:
# scikit-learn (imported as "sklearn") is the standard Python library for
# classical machine learning. We take two scoring functions from it:
#   f1_score          computes the F1 score
#   confusion_matrix  builds a table of "what it was" vs "what the model said"
from sklearn.metrics import f1_score, confusion_matrix

In [ ]:
# Compute the token-level F1 score.
#   labels            the correct answers (first argument)
#   results           the model's predictions (second argument)
#   labels=[1,2]      only score label 1 (B) and label 2 (I). Label 0 (O) is
#                     left out on purpose: most tokens are O, they are easy,
#                     and including them would make the score look better
#                     than the model really is at finding aspect terms.
#                     (Yes, the word "labels" appears twice: once as our
#                     variable, once as the name of the function's argument.)
#   average='macro'   compute F1 for B and for I separately, then take the
#                     plain average of the two, so both count equally.
# The result is a number between 0 (useless) and 1 (perfect).
f1_score(labels,results,labels =[1,2],average='macro')

In [ ]:
# Build the confusion matrix: a 3x3 table that shows exactly WHICH mistakes
# the model makes.
#   ROWS    = the correct label     (row 0 = O, row 1 = B, row 2 = I)
#   COLUMNS = the predicted label   (col 0 = O, col 1 = B, col 2 = I)
# Each cell counts tokens. How to read it:
#   - The diagonal (top-left to bottom-right) holds the CORRECT predictions.
#   - Everything off the diagonal is an error. Examples:
#       row 1, column 0: a real B that the model called O (a missed term)
#       row 0, column 1: a real O that the model called B (a false alarm)
#       row 2, column 1: a real I that the model called B (wrong position)
# A good model has big numbers on the diagonal and small ones elsewhere.
confusion_matrix(labels,results)